# Study 955 — ADR Catch-Up 🌏

**When Tokyo closed thirteen hours ago, does the New York listing still owe it a move?**

Toyota trades in Tokyo and, as an ADR, in New York. Tokyo shuts at 02:00 New York time;
London and Frankfurt at 11:30, *inside* the US session. So for part of the world the ADR
gets a whole American day to price news its home market already priced and went home on.
The trading-desk folklore: watch the home index and the currency overnight, and you know
which way the ADR is going.

We test that on **eight** cross-listed ADRs (TM, SONY, SAP, NVO, SHEL, BP, HSBC, RIO) against
their home indices and currencies, **2004-01-05 → 2026-06-30** (41,272 name-days),
with one execution lag and real costs.

*Real numbers below are the frozen headline (`docs/results.md`, Fingerprint `a564d6b737e5`);
the live cells run the fast synthetic control and are labelled as such. As-of 2026-06-30.*

> ⚠️ **The honest limit, up front.** Real catch-up is an *intraday* effect. A daily-close
> tape sees only close-to-close, so what we can measure is the residue that survives to the
> ADR's own close — a weaker question than "what happens in the US session". Everything
> here answers the weaker question, and says so.


## 1. The idea

The ADR and the home share are the same company. If Tokyo rallied 2% overnight and Toyota's New York line has not moved 2%, somebody is about to be paid. The only question is *when* — and whether anything is still owed by the time New York shuts.

In [1]:
R = dict(b0=0.5761, b1=0.0209, t_b1=1.15, jp_b1=0.0809, jp_t_b1=3.25, uk_b1=-0.0479, uk_t_b1=-1.8)
print('Across all eight ADRs, how much of YESTERDAY\'s home move is still owed?')
print('  loading on yesterday: %+.4f   (t = %+.2f)  -> nothing'
      % (R['b1'], R['t_b1']))
print()
print('Split by where the home market is:')
print('  Japan  (Tokyo shuts 02:00 New York time) : %+.4f  (t = %+.2f)'
      % (R['jp_b1'], R['jp_t_b1']))
print('  UK     (London shuts 11:30, mid-session) : %+.4f  (t = %+.2f)'
      % (R['uk_b1'], R['uk_t_b1']))

Across all eight ADRs, how much of YESTERDAY's home move is still owed?
  loading on yesterday: +0.0209   (t = +1.15)  -> nothing

Split by where the home market is:
  Japan  (Tokyo shuts 02:00 New York time) : +0.0809  (t = +3.25)
  UK     (London shuts 11:30, mid-session) : -0.0479  (t = -1.80)


## 2. The clock predicts who has a lag

Pooled over all eight names, yesterday's home move is worth **+0.0209** with *t* = +1.15 — nothing at all. But split them by *when their home market closes* and a clean pattern falls out.

- **Japan** — Tokyo shuts thirteen hours before the ADR does. Yesterday's move still loads **+0.0809** (*t* = **+3.25**). Both names on their own: Toyota *t* = +3.08, Sony *t* = +2.75.
- **UK** — London shuts at 11:30 New York time, *inside* the US session, so most of its move is already in the ADR's own day. Loading: -0.0479 (*t* = -1.80) — nothing left over, exactly as the mechanism says.

That is not a fishing expedition finding a lucky subgroup. It is the theory's own prediction — a lag exists where the markets do not overlap — coming true, and failing to appear where they do.

## 3. And Japan's lag is not a fluke of one decade

| Block | loading on yesterday | *t* |
|---|--:|--:|
| 2004-2009 | +0.0806 | +1.74 |
| 2010-2014 | +0.1119 | +2.35 |
| 2015-2020 | +0.0540 | +1.15 |
| 2021-2026 | +0.0658 | +2.10 |
| **2004–2014 half** | **+0.0935** | **+2.60** |
| **2015–2026 half** | **+0.0598** | **+2.25** |

Positive in every block, both halves, and still alive in the last five years. So far this looks like a finding. Section 3½ is where it stops looking like one.

## 3½. Who actually owns that number?

A *loading* should be boring and linear: if Toyota's New York line repays a fixed slice of whatever Tokyo did last night, then throwing away the wildest nights should barely move the estimate. Here is what happens when we do.

| what we removed | Japan's loading | *t* |
|---|--:|--:|
| full sample | +0.0809 | +3.25 |
| trim >99.5th pct | +0.0169 | +0.67 |
| trim >99th pct | +0.0307 | +1.52 |
| trim >97.5th pct | +0.0358 | +1.75 |
| trim >95th pct | +0.0418 | +1.82 |
| winsorize @99th | +0.0603 | +2.62 |
| winsorize @97.5th | +0.0567 | +2.58 |

Deleting **48 rows** — half of one per cent of the sample — takes the study's only significant number from +0.0809 (*t* = +3.25) to +0.0169 (*t* = +0.67). For comparison, the identical knife on a *simulated* market where a catch-up lag really was planted, and planted linearly, moves the estimate by about one per cent (next section's live cell prints it). So the honest reading is: on the 0.5% of nights when Tokyo moved enormously there is something; on an ordinary night there is nothing.

The plainest version of the same question — sort the days by what Tokyo did last night, and see what the ADR paid next:

| yesterday in Tokyo | what the ADR paid next |
|---|--:|
| Q1 (-1.92%) | +0.00 bp |
| Q2 (-0.53%) | +4.81 bp |
| Q3 (+0.05%) | +6.38 bp |
| Q4 (+0.61%) | +1.23 bp |
| Q5 (+1.91%) | +1.60 bp |

**Best minus worst: +1.60 basis points**, and not even in a straight line. There is no gradient to bet on.

## 4. So why isn't this a trade?

Three reasons, and each one is fatal on its own.

**(a) That coefficient is not the one you can bet on.** It is measured while *also* knowing tomorrow's home move — which you don't. Strip that out and the bettable number in Japan is **+0.0331** (*t* = +1.43), about a third the size. The reason is dull and arithmetic: the Nikkei-in-dollars is itself negatively autocorrelated (-0.165), which inflates the first number relative to the second.

**(b) It is tiny against the noise, and it is not there most nights.** Read at face value it is an 8-basis-point payback on a 1% Tokyo move, on a stock that moves ~30% a year on its own news — and section 3½ showed most of even that comes from the wildest half-per-cent of nights. You would need thousands of bets to see it, and you get one a day.

**(c) The costs eat it before you start.** The book has to be re-set every session — it turns over **108% of the portfolio per day**. It breaks even at **0.85 basis points** of one-way cost in Japan and at **-0.34 bps** across all eight, which is to say it never breaks even at all.

In [2]:
R = dict(cu_gross=-0.92, cu_sharpe=-0.049, cu_net=-14.81, cu_be=-0.34,
         jp_cu_gross=2.3, jp_cu_sharpe=0.09, jp_cu_be=0.85,
         rv_gross=5.28, rv_sharpe=0.326)
print('Buy the ADRs whose home market rose, sell the ones that fell:')
print('  all eight, before costs : %+6.2f%% a year   Sharpe %+.3f'
      % (R['cu_gross'], R['cu_sharpe']))
print('  Japan only, before costs: %+6.2f%% a year   Sharpe %+.3f'
      % (R['jp_cu_gross'], R['jp_cu_sharpe']))
print('  all eight, after 5bp    : %+6.2f%% a year'  % R['cu_net'])
print()
print('A control book that never looks at the home market at all,')
print('and just fades whatever the ADR did yesterday:')
print('  before costs            : %+6.2f%% a year   Sharpe %+.3f  <- better!'
      % (R['rv_gross'], R['rv_sharpe']))

Buy the ADRs whose home market rose, sell the ones that fell:
  all eight, before costs :  -0.92% a year   Sharpe -0.049
  Japan only, before costs:  +2.30% a year   Sharpe +0.090
  all eight, after 5bp    : -14.81% a year

A control book that never looks at the home market at all,
and just fades whatever the ADR did yesterday:
  before costs            :  +5.28% a year   Sharpe +0.326  <- better!


## 5. The line that settles it

A dumb control book — fade whatever the ADR did yesterday, never open the home tape at all — earns **+5.28%/yr** gross (Sharpe +0.326), while the home-informed catch-up book earns **-0.92%/yr** (Sharpe -0.049). In tradable form, the foreign market's information is a *drag*. Every one of those Sharpes has a bootstrap confidence interval straddling zero anyway.

## 6. Live check — the machinery is not broken

*The cell below is the **synthetic** control — a simulated panel with a known planted answer, run live so you can see the machinery is unbiased. It is not the real tape and carries no part of the verdict.*

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from adr_catchup import data, strategy as st
syn = data.synthetic_panel(signal_strength=1.0, seed=955)[0]
planted = st.synthetic_detect(syn)
null    = st.synthetic_detect(data.synthetic_panel(signal_strength=0.0, seed=955)[0])
print('SYNTHETIC world where 40 pct of the home move really does arrive a day late:')
print('   detected lag %+.4f (t %+.1f)   book Sharpe %+.2f  -> the detector fires'
      % (planted['beta_lag'], planted['t_lag'], planted['gross_sharpe']))
print('SYNTHETIC world where the ADR prices everything the same day:')
print('   detected lag %+.4f (t %+.2f)  -> and stays quiet'
      % (null['beta_lag'], null['t_lag']))
print()
print('And the calibration for section 3.5 - the same knife, on the SYNTHETIC')
print('world where the lag is real AND linear:')
tail = st.tail_sensitivity(syn, quantiles=(0.99, 0.95))
for _, r in tail.iterrows():
    label = 'full sample' if r['mode'] == 'full' else '%s > %.3f' % (r['mode'], r['q'])
    print('   %-18s lag %+.4f (t %+.1f)' % (label, r['beta_lag'], r['t_lag']))
print('   -> a genuine linear lag does not care. Japan lost 79 pct of its size.')

SYNTHETIC world where 40 pct of the home move really does arrive a day late:
   detected lag +0.2510 (t +47.8)   book Sharpe +7.54  -> the detector fires
SYNTHETIC world where the ADR prices everything the same day:
   detected lag +0.0046 (t +0.87)  -> and stays quiet

And the calibration for section 3.5 - the same knife, on the SYNTHETIC
world where the lag is real AND linear:


   full sample        lag +0.2510 (t +47.8)
   trim > 0.990       lag +0.2489 (t +45.4)
   winsor > 0.990     lag +0.2529 (t +47.7)
   trim > 0.950       lag +0.2511 (t +40.4)
   winsor > 0.950     lag +0.2614 (t +47.4)
   -> a genuine linear lag does not care. Japan lost 79 pct of its size.


> 🔬 **For the quants.** The same generator also plants pure bid-ask bounce with *zero* stale information, and that fires the residual-reversal rule hard while leaving the lagged-home-move test silent. That is why this study never lets a negative residual slope stand as evidence of catch-up — see notebook 02.

## Verdict

- **Signal — Weak.** The blanket folklore fails: pooled over eight ADRs, yesterday's home move is worth +0.0209 (*t* = +1.15), and the little that is there comes from the *currency*, which never closes, rather than the index, which does. Japan — the one home market that shuts long before New York — does carry **+0.0809** (*t* = **+3.25**) where the clock says it should, and the UK, closing mid-session, carries nothing. But that number is owned by the tail: drop the 48 wildest nights and it falls to +0.0169 (*t* = +0.67), where a simulated *linear* lag would not have moved at all, and sorting the days by yesterday's Tokyo move pays +1.60 bp best-minus-worst with no gradient. A whiff on extreme nights, on two survivorship-picked names — not a result.
- **Tradability — Mirage.** Negative gross across all eight, break-even at 0.85 bps in Japan, 108% daily turnover, and beaten by a control that ignores the home market entirely. Not a trade.